# One dataset, several views

Inspect the complete sample before mapping it. All 24 observations were generated by `scripts/generate_data.py`; approximate neighborhood locations give context while hour, offsets and values are invented.

Run top to bottom. Only standard-library Python is needed.

In [ ]:
import json
from pathlib import Path
from collections import Counter
from selection import select_features, feature_collection, export_geojson

dataset = json.loads(Path('data/nyc-observations.geojson').read_text())
print(json.dumps(dataset['metadata'], indent=2))

In [ ]:
counts = Counter(f['properties']['hour'] for f in dataset['features'])
for hour, count in sorted(counts.items()):
    print(f'{hour:02d}:00  {count} observations')
assert counts == {8: 6, 12: 6, 18: 6, 21: 6}

## Choose an hour

Change `hour` to 8, 12, 18 or 21. Use `'all'` for the entire sample and `0` for a valid empty selection. The shared function returns copies, preserving the input data.

In [ ]:
hour = 8
selected = feature_collection(dataset, hour)
print('Selected:', selected['metadata']['feature_count'])
for feature in selected['features']:
    p = feature['properties']
    print(feature['id'], p['name'], p['value'], feature['geometry']['coordinates'])

## Verify the empty case

Zero observations should clear an app's observation layer. Map tiles or the scene grid can remain.

In [ ]:
empty = feature_collection(dataset, 0)
assert empty['type'] == 'FeatureCollection'
assert empty['features'] == []
assert empty['metadata']['feature_count'] == 0
print(export_geojson(dataset, 0))

## Write an actual handoff file

The file below is created inside JupyterLite's virtual filesystem. To move it out, find it in the file browser, right-click and **Download**. Then upload/import it into another app. It does not automatically appear in a PyScript lab or another JupyterLite deployment.

In [ ]:
path = Path(f'synthetic-nyc-hour-{hour}.geojson')
path.write_text(export_geojson(dataset, hour))
round_trip = json.loads(path.read_text())
assert len(round_trip['features']) == len(selected['features'])
assert all(f['properties']['simulated'] for f in round_trip['features'])
print('Created:', path, '— download it from the file browser.')

## Compare with the maps

Set the same hour in the [2D map]({{HUB_BASE}}/apps/leaflet/), [3D columns]({{HUB_BASE}}/apps/deck/) and [3D scene]({{HUB_BASE}}/apps/scene/). Compare the displayed count and downloaded feature IDs with this notebook.

Continue to the [GeoLibre / RustGIS / JupyterGIS import exercise]({{HUB_BASE}}/integrations/). The CRS is EPSG:4326 and coordinate order is longitude, latitude. Metadata is a GeoJSON foreign member; the per-feature `simulated` field carries the label even if a tool drops collection metadata.